<div style="background-color: #0b1329; border-radius: 12px; padding: 30px; text-align: center; color: #ffffff; font-family: 'Inter', sans-serif; box-shadow: 0 10px 30px rgba(0,0,0,0.2); margin-bottom: 25px;">
    <span style="background-color: #172554; color: #3b82f6; border-radius: 20px; padding: 6px 16px; font-size: 13px; font-weight: bold; letter-spacing: 1px; text-transform: uppercase;">YBS3259 - MAKİNE ÖĞRENMESİ FİNAL PROJESİ</span>
    <h1 style="font-size: 34px; font-weight: 800; margin-top: 20px; margin-bottom: 10px; color: #ffffff; line-height: 1.2;">Çalışan Kaybı (Employee Churn)<br>Tahmin Modeli</h1>
    <p style="font-size: 16px; color: #94a3b8; font-weight: 400; margin-top: 10px; margin-bottom: 20px;">Makine Öğrenmesi Tabanlı İnsan Kaynakları Karar Destek Sistemi</p>
    <hr style="border-color: #1e293b; margin: 15px 0;">
    <p style="color: #cbd5e1; font-size: 14px; line-height: 1.5; margin: 0;">Bu notebook, şirket çalışanlarının istifa riskini önceden tahmin etmek ve İK departmanının elde tutma bütçesini en verimli şekilde kullanmasını sağlamak amacıyla <b>CRISP-DM</b> standartlarına uygun olarak tasarlanmıştır.</p>
</div>

<div style="background-color: #0b1329; border-left: 6px solid #3b82f6; border-radius: 8px; padding: 20px; color: #ffffff; font-family: 'Inter', sans-serif; margin-top: 20px; margin-bottom: 15px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: #ffffff; font-size: 20px; font-weight: 700;">1. Business Understanding (İş Problemi & Karar Bağlamı)</h2>
</div>

<div style="background-color: #1e293b; border-left: 5px solid #3b82f6; padding: 18px; border-radius: 6px; margin: 15px 0; color: #ffffff; font-family: 'Inter', sans-serif;">
    <h3 style="margin-top: 0; color: #3b82f6; font-family: 'Inter', sans-serif; font-size: 16px; font-weight: 700; margin-bottom: 8px;">🎯 İş Problemi</h3>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">Şirket çalışanlarının istifa etmesi (çalışan kaybı/churn), yeni işe alım maliyetleri, oryantasyon süreleri ve departman içi bilgi kaybı nedeniyle ciddi maliyetler doğurmaktadır. İK departmanı için <b>ortalama bir çalışanın ayrılma maliyeti $15,000</b> olarak hesaplanmıştır.</p>
</div>

<div style="background-color: #1e293b; border-left: 5px solid #10b981; padding: 18px; border-radius: 6px; margin: 15px 0; color: #ffffff; font-family: 'Inter', sans-serif;">
    <h3 style="margin-top: 0; color: #10b981; font-family: 'Inter', sans-serif; font-size: 16px; font-weight: 700; margin-bottom: 8px;">💡 Karar Bağlamı ve Müdahale Stratejisi</h3>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">İK departmanı, ayrılma riski yüksek olan çalışanları önceden belirleyebilir ve <b>çalışan başına $3,000 elde tutma bütçesi</b> (prim, terfi, rotasyon gibi aksiyonlar) kullanarak bu çalışanların <b>%80 olasılıkla şirkette kalmasını</b> sağlayabilir.</p>
</div>

<div style="background-color: #1e293b; border-left: 5px solid #ec4899; padding: 18px; border-radius: 6px; margin: 15px 0; color: #ffffff; font-family: 'Inter', sans-serif;">
    <h3 style="margin-top: 0; color: #ec4899; font-family: 'Inter', sans-serif; font-size: 16px; font-weight: 700; margin-bottom: 8px;">📊 Başarı Kriterleri ve Risk Analizi</h3>
    <ul style="margin: 0; color: #cbd5e1; padding-left: 20px; font-size: 14px; line-height: 1.6;">
        <li><b>Recall (Duyarlılık):</b> En kritik metriğimizdir. Ayrılacak kişileri gözden kaçırmanın maliyeti ($15,000) yüksek olduğu için False Negative'leri minimize etmeliyiz.</li>
        <li><b>Precision (Kesinlik):</b> Bütçeyi verimli kullanmak için gereksiz yere elde tutma aksiyonu uygulanacak kişileri (False Positive) sınırlamalıyız.</li>
        <li><b>Finansal Net Tasarruf:</b> Modeli devreye aldığımızda İK'nın elde edeceği toplam finansal tasarruf ana başarı kriterimizdir.</li>
    </ul>
</div>

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit, cross_val_score, GroupKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

print("✅ Gerekli kütüphaneler başarıyla yüklendi.")

<div style="background-color: #0b1329; border-left: 6px solid #2e86ab; border-radius: 8px; padding: 20px; color: #ffffff; font-family: 'Inter', sans-serif; margin-top: 20px; margin-bottom: 15px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: #ffffff; font-size: 20px; font-weight: 700; margin-bottom: 10px;">2. Data Understanding (Veriyi Tanıma & Kalite Kontrolü)</h2>
    <p style="margin: 0; color: #94a3b8; font-size: 14px; line-height: 1.5;">Veri kümesini yükleyerek genel profilini, eksik/aykırı değerleri ve öznitelik türlerini inceleyelim.</p>
</div>

In [ ]:
# Veriyi oku
df = pd.read_csv('../data/raw/veri_seti.csv')
print(f"Veri Boyutu: {df.shape[0]} satır, {df.shape[1]} sütun")
print("\nEksik Değer Durumu:")
print(df.isnull().sum())
df.info()

<div style="background-color: #1e293b; border-left: 4px solid #ef4444; border-radius: 6px; padding: 15px 20px; color: #ffffff; font-family: 'Inter', sans-serif; margin-top: 15px; margin-bottom: 10px;">
    <h3 style="margin: 0; color: #ffffff; font-size: 16px; font-weight: 600; margin-bottom: 8px;">2.1. Target (Hedef Değişken) Analizi</h3>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">Hedef değişkenimiz olan <code>STATUS</code> (ACTIVE / TERMINATED) sınıflarının oranını inceleyelim.</p>
</div>

In [ ]:
target_counts = df['STATUS'].value_counts()
target_ratios = df['STATUS'].value_counts(normalize=True) * 100

fig = px.bar(
    x=target_counts.index,
    y=target_ratios.values,
    color=target_counts.index,
    color_discrete_sequence=['#2E86AB', '#C73E1D'],
    title="Hedef Değişken Sınıf Dağılım Oranları (%)",
    labels={'x': 'Status', 'y': 'Oran (%)'}
)
fig.show()

<div style="background-color: #0f172a; border-left: 5px solid #fbbf24; padding: 15px; border-radius: 6px; margin: 15px 0; color: #ffffff; font-family: 'Inter', sans-serif;">
    <h4 style="margin-top: 0; color: #fbbf24; font-weight: 700; font-size: 15px;">🔍 Veri Analisti Yorumu</h4>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">Sınıf dağılımında görüldüğü üzere, çalışanların <b>%97.01'i ACTIVE</b>, yalnızca <b>%2.99'u TERMINATED</b> durumundadır. Bu durum veri kümesinde aşırı sınıf dengesizliği (class imbalance) olduğunu gösterir. Modelleme aşamasında modellerin bu durumdan etkilenmesini önlemek için <code>class_weight='balanced'</code> gibi yöntemler kullanılmalıdır.</p>
</div>

<div style="background-color: #1e293b; border-left: 4px solid #3b82f6; border-radius: 6px; padding: 15px 20px; color: #ffffff; font-family: 'Inter', sans-serif; margin-top: 15px; margin-bottom: 10px;">
    <h3 style="margin: 0; color: #ffffff; font-size: 16px; font-weight: 600; margin-bottom: 8px;">2.1b. İş Birimi, Cinsiyet ve Demografik Kıyaslamalar</h3>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">İstifa eden ve aktif kalan çalışanların demografik ve operasyonel kırılımlarını inceleyelim.</p>
</div>

In [ ]:
# Demografik ortalamalar (İstifa vs Aktif)
stats_by_status = df.groupby('STATUS')[['age', 'length_of_service']].mean()
print("--- ORTALAMA YAŞ VE KIDEM DEĞERLERİ ---")
print(stats_by_status)

# İş Birimi bazlı istifa oranları
bu_churn = df.groupby('BUSINESS_UNIT')['target'].mean() * 100
print("\n--- İŞ BİRİMİ BAZLI İSTİFA ORANLARI (%) ---")
print(bu_churn)

# Cinsiyet bazlı istifa oranları
gender_churn = df.groupby('gender_full')['target'].mean() * 100
print("\n--- CİNSİYET BAZLI İSTİFA ORANLARI (%) ---")
print(gender_churn)

<div style="background-color: #0f172a; border-left: 5px solid #fbbf24; padding: 15px; border-radius: 6px; margin: 15px 0; color: #ffffff; font-family: 'Inter', sans-serif;">
    <h4 style="margin-top: 0; color: #fbbf24; font-weight: 700; font-size: 15px;">🔍 Veri Analisti Yorumu</h4>
    <ul style="margin: 0; color: #cbd5e1; padding-left: 20px; font-size: 14px; line-height: 1.6;">
        <li><b>Yaş Dağılımı:</b> İstifa edenlerin yaş ortalaması (51.46), aktif çalışanlarınkinden (41.79) belirgin şekilde yüksektir. Bu durum, istifaların daha çok emeklilik veya kıdemli seviyelerdeki çalışan hareketliliğinden kaynaklandığını göstermektedir.</li>
        <li><b>İş Birimi (Business Unit):</b> Genel Müdürlük (HEADOFFICE) biriminde istifa oranı %11.79 iken, Mağazalarda (STORES) bu oran %2.89'dur. HEADOFFICE'teki 4 kat daha yüksek churn oranı, İK için kritik bir alarm durumudur ve özel odaklanma gerektirir.</li>
        <li><b>Cinsiyet Kırılımı:</b> Kadın çalışanların istifa oranı (%3.53), erkek çalışanlarınkine kıyasla (%2.40) biraz daha yüksektir.</li>
    </ul>
</div>

<div style="background-color: #1e293b; border-left: 4px solid #3b82f6; border-radius: 6px; padding: 15px 20px; color: #ffffff; font-family: 'Inter', sans-serif; margin-top: 15px; margin-bottom: 10px;">
    <h3 style="margin: 0; color: #ffffff; font-size: 16px; font-weight: 600; margin-bottom: 8px;">2.1c. Gelişmiş İstatistiksel Hipotez Testleri</h3>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">Gözlemlediğimiz demografik farkların (Yaş, Kıdem, İş Birimi ve Cinsiyet) istatistiksel olarak anlamlı olup olmadığını bilimsel testlerle doğrulayalım.</p>
</div>

In [ ]:
from scipy import stats

# 1. İki Bağımsız Örneklem T-Testi (Yaş)
active_age = df[df['STATUS'] == 'ACTIVE']['age']
term_age = df[df['STATUS'] == 'TERMINATED']['age']
t_age, p_age = stats.ttest_ind(active_age, term_age, equal_var=False)
print(f"Yaş için T-İstatistiği: {t_age:.4f}, p-değeri: {p_age:.4e}")

# 2. İki Bağımsız Örneklem T-Testi (Kıdem)
active_los = df[df['STATUS'] == 'ACTIVE']['length_of_service']
term_los = df[df['STATUS'] == 'TERMINATED']['length_of_service']
t_los, p_los = stats.ttest_ind(active_los, term_los, equal_var=False)
print(f"Kıdem Süresi için T-İstatistiği: {t_los:.4f}, p-değeri: {p_los:.4e}")

# 3. Ki-Kare İlişki Testi (İş Birimi)
ct_bu = pd.crosstab(df['BUSINESS_UNIT'], df['STATUS'])
chi2_bu, p_bu, _, _ = stats.chi2_contingency(ct_bu)
print(f"İş Birimi Ki-Kare p-değeri: {p_bu:.4e}")

# 4. Ki-Kare İlişki Testi (Cinsiyet)
ct_gen = pd.crosstab(df['gender_full'], df['STATUS'])
chi2_gen, p_gen, _, _ = stats.chi2_contingency(ct_gen)
print(f"Cinsiyet Ki-Kare p-değeri: {p_gen:.4e}")

<div style="background-color: #0f172a; border-left: 5px solid #fbbf24; padding: 15px; border-radius: 6px; margin: 15px 0; color: #ffffff; font-family: 'Inter', sans-serif;">
    <h4 style="margin-top: 0; color: #fbbf24; font-weight: 700; font-size: 15px;">🔍 İstatistiksel Değerlendirme Yorumu</h4>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">Tüm test sonuçlarında p-değerleri <b>alpha = 0.05</b> anlamlılık eşiğinin son derece altındadır. Bu durum:<br>
    - İstifa edenler ile aktif çalışanlar arasındaki yaş farkının ve kıdem süresi farkının rastlantısal olmadığını,<br>
    - İş Birimi (HEADOFFICE vs STORES) ile Cinsiyet değişkenlerinin istifa kararı üzerinde istatistiksel olarak son derece belirleyici (anlamlı) olduğunu kanıtlar.</p>
</div>

<div style="background-color: #1e293b; border-left: 4px solid #3b82f6; border-radius: 6px; padding: 15px 20px; color: #ffffff; font-family: 'Inter', sans-serif; margin-top: 15px; margin-bottom: 10px;">
    <h3 style="margin: 0; color: #ffffff; font-size: 16px; font-weight: 600; margin-bottom: 8px;">2.1d. Lokasyon ve Pozisyon Bazlı Risk Dağılımı</h3>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">İstifa oranlarının hangi şehirler ve hangi iş unvanlarında yoğunlaştığını inceleyelim.</p>
</div>

In [ ]:
# En yüksek istifa oranına sahip şehirler
city_churn = df.groupby('city_name')['target'].agg(['count', 'mean'])
city_churn = city_churn[city_churn['count'] >= 30].sort_values('mean', ascending=False).head(10)
city_churn['mean'] = city_churn['mean'] * 100

fig_city = px.bar(
    city_churn, x=city_churn.index, y="mean", text_auto='.1f',
    title="En Yüksek İstifa Oranına Sahip 10 Şehir (En az 30 Kayıt Olanlar)",
    labels={'mean': 'İstifa Oranı (%)', 'city_name': 'Şehir'},
    color="mean", color_continuous_scale="Oranges"
)
fig_city.show()

# En yüksek istifa oranına sahip unvanlar
job_churn = df.groupby('job_title')['target'].agg(['count', 'mean'])
job_churn = job_churn.sort_values('mean', ascending=False).head(10)
job_churn['mean'] = job_churn['mean'] * 100

fig_job = px.bar(
    job_churn, x="mean", y=job_churn.index, orientation='h', text_auto='.1f',
    title="En Yüksek İstifa Oranına Sahip 10 Unvan",
    labels={'mean': 'İstifa Oranı (%)', 'job_title': 'Unvan'},
    color="mean", color_continuous_scale="Reds"
)
fig_job.show()


<div style="background-color: #0f172a; border-left: 5px solid #fbbf24; padding: 15px; border-radius: 6px; margin: 15px 0; color: #ffffff; font-family: 'Inter', sans-serif;">
    <h4 style="margin-top: 0; color: #fbbf24; font-weight: 700; font-size: 15px;">🔍 Coğrafi ve Pozisyonel Analiz Yorumu</h4>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">Analizde görüldüğü üzere <b>New Westminister (%17.3)</b> ve <b>Pitt Meadows (%15.8)</b> şehirlerindeki istifa oranları, %2.99 olan şirket ortalamasının neredeyse 5-6 katıdır. Pozisyonel olarak da Director düzeyindeki rollerde istifa oranının %25'lere çıkması, üst yönetim düzeyinde ciddi bir elde tutma stratejisi ihtiyacına işaret eder.</p>
</div>

<div style="background-color: #1e293b; border-left: 4px solid #6a994e; border-radius: 6px; padding: 15px 20px; color: #ffffff; font-family: 'Inter', sans-serif; margin-top: 15px; margin-bottom: 10px;">
    <h3 style="margin: 0; color: #ffffff; font-size: 16px; font-weight: 600; margin-bottom: 8px;">2.2. Sayısal Değişkenlerin ve Aykırı Değerlerin Analizi</h3>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">Yaş (<code>age</code>) ve Hizmet Süresi (<code>length_of_service</code>) değişkenlerinin dağılımlarını ve aykırı değerlerini Box-plot ile inceleyelim.</p>
</div>

In [ ]:
fig_age = px.box(df, y="age", title="Çalışan Yaş Dağılımı ve Aykırı Değerler", color_discrete_sequence=['#2E86AB'])
fig_age.show()

fig_service = px.box(df, y="length_of_service", title="Hizmet Süresi Dağılımı ve Aykırı Değerler", color_discrete_sequence=['#6A994E'])
fig_service.show()

<div style="background-color: #0f172a; border-left: 5px solid #fbbf24; padding: 15px; border-radius: 6px; margin: 15px 0; color: #ffffff; font-family: 'Inter', sans-serif;">
    <h4 style="margin-top: 0; color: #fbbf24; font-weight: 700; font-size: 15px;">🔍 Veri Analisti Yorumu</h4>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">Yaş dağılımı 19 ile 65 yaş arasında homojen bir dağılım göstermektedir ve belirgin bir aykırı değer bulunmamaktadır. Hizmet süresi ise 0 ile 26 yıl arasında değişmektedir. Çeyreklikler arası aralıkta (IQR) aykırı değer olabilecek aşırı uç noktalar görülmemiştir.</p>
</div>

<div style="background-color: #1e293b; border-left: 4px solid #ef4444; border-radius: 6px; padding: 15px 20px; color: #ffffff; font-family: 'Inter', sans-serif; margin-top: 15px; margin-bottom: 10px;">
    <h3 style="margin: 0; color: #ffffff; font-size: 16px; font-weight: 600; margin-bottom: 8px;">2.3. Departman Bazlı İstifa Oranları</h3>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">Farklı departmanlardaki çalışan istifa oranlarını karşılaştıralım.</p>
</div>

In [ ]:
dept_churn = df.groupby('department_name')['target'].mean().reset_index()
dept_churn['target'] = dept_churn['target'] * 100
dept_churn = dept_churn.sort_values('target', ascending=False)

fig_dept = px.bar(
    dept_churn, x="target", y="department_name", orientation="h",
    title="Departman Bazlı İstifa Oranları (%)",
    color="target", color_continuous_scale="Reds",
    labels={'target': 'İstifa Oranı (%)', 'department_name': 'Departman'}
)
fig_dept.show()

<div style="background-color: #0f172a; border-left: 5px solid #fbbf24; padding: 15px; border-radius: 6px; margin: 15px 0; color: #ffffff; font-family: 'Inter', sans-serif;">
    <h4 style="margin-top: 0; color: #fbbf24; font-weight: 700; font-size: 15px;">🔍 Veri Analisti Yorumu</h4>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">Departman bazında istifa oranları incelendiğinde bazı kritik birimlerde istifa oranlarının belirgin şekilde yüksek olduğu görülebilir. Bu durum İK için hedefli aksiyon alma imkanı sunar.</p>
</div>

<div style="background-color: #1e293b; border-left: 4px solid #2e86ab; border-radius: 6px; padding: 15px 20px; color: #ffffff; font-family: 'Inter', sans-serif; margin-top: 15px; margin-bottom: 10px;">
    <h3 style="margin: 0; color: #ffffff; font-size: 16px; font-weight: 600; margin-bottom: 8px;">2.4. Kıdem Yılına Göre İstifa Oranları</h3>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">Çalışanların şirkette geçirdikleri yıla (kıdem/hizmet süresi) göre istifa eğilimlerini inceleyelim.</p>
</div>

In [ ]:
service_churn = df.groupby('length_of_service')['target'].mean().reset_index()
service_churn['target'] = service_churn['target'] * 100

fig_service_churn = px.line(
    service_churn, x="length_of_service", y="target", markers=True,
    title="Kıdem Yılına Göre İstifa Oranları (%)",
    labels={'target': 'İstifa Oranı (%)', 'length_of_service': 'Hizmet Süresi (Yıl)'}
)
fig_service_churn.show()

<div style="background-color: #0f172a; border-left: 5px solid #fbbf24; padding: 15px; border-radius: 6px; margin: 15px 0; color: #ffffff; font-family: 'Inter', sans-serif;">
    <h4 style="margin-top: 0; color: #fbbf24; font-weight: 700; font-size: 15px;">🔍 Veri Analisti Yorumu</h4>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">Kıdem süresi analizinde belirli hizmet yıllarında (örn. ilk yıllarda veya belirli tecrübe eşiklerinde) istifa oranının pik yaptığı görülebilir. Bu grafik İK'ya kariyer aşamalarına göre özelleştirilmiş retention programları tasarlama içgörüsü sağlar.</p>
</div>

<div style="background-color: #1e293b; border-left: 4px solid #6a994e; border-radius: 6px; padding: 15px 20px; color: #ffffff; font-family: 'Inter', sans-serif; margin-top: 15px; margin-bottom: 10px;">
    <h3 style="margin: 0; color: #ffffff; font-size: 16px; font-weight: 600; margin-bottom: 8px;">2.5. Korelasyon Analizi</h3>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">Sayısal değişkenlerin kendi aralarındaki korelasyonunu inceleyelim.</p>
</div>

In [ ]:
numerical_cols = ['age', 'length_of_service', 'STATUS_YEAR']
corr_matrix = df[numerical_cols].corr()
fig_corr = px.imshow(
    corr_matrix, 
    text_auto=".3f", 
    title="Sayısal Özniteliklerin Korelasyon Matrisi",
    color_continuous_scale="RdBu_r"
)
fig_corr.show()

<div style="background-color: #0f172a; border-left: 5px solid #fbbf24; padding: 15px; border-radius: 6px; margin: 15px 0; color: #ffffff; font-family: 'Inter', sans-serif;">
    <h4 style="margin-top: 0; color: #fbbf24; font-weight: 700; font-size: 15px;">🔍 Veri Analisti Yorumu</h4>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">Yaş ve hizmet süresi arasında <b>0.58</b> düzeyinde pozitif yönlü, orta şiddette bir ilişki vardır. Bu durum yaşlandıkça hizmet süresinin artma eğilimini gösterir. Çoklu doğrusal bağlantı (multicollinearity) riski taşıyacak derecede yüksek (&gt;0.80) bir ilişki bulunmamaktadır.</p>
</div>

<div style="background-color: #0b1329; border-left: 6px solid #d97706; border-radius: 8px; padding: 20px; color: #ffffff; font-family: 'Inter', sans-serif; margin-top: 20px; margin-bottom: 15px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: #ffffff; font-size: 20px; font-weight: 700; margin-bottom: 10px;">3. Data Preparation (Veri Hazırlama & Sızıntı Engelleme)</h2>
    <div style="background-color: #1e1b4b; border-left: 5px solid #f59e0b; padding: 15px; border-radius: 6px; margin-top: 10px; color: #ffffff;">
        <h3 style="margin-top: 0; color: #fbbf24; font-size: 16px; font-weight: 700; margin-bottom: 8px;">⚠️ Veri Sızıntısı (Data Leakage) Nedir ve Nasıl Engellenir?</h3>
        <p style="color: #cbd5e1; font-size: 14px; line-height: 1.5;">Bu veri kümesinde aynı çalışanın farklı yıllardaki kayıtları yer almaktadır (boylamsal/panel veri). Rastgele bir satır bazlı bölünme yaparsak, aynı çalışanın 2012 yılı verisi eğitim setine, 2013 yılı verisi test setine düşebilir. Bu durum modelin çalışanı ezberlemesine (leakage) yol açar.</p>
        <p style="margin-bottom: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;"><b>Bunu engellemek için:</b><br>
        1. Çalışan ID bazlı gruplama yaparak bölünme sağlayacağız (<code>GroupShuffleSplit</code>).<br>
        2. <code>terminationdate_key</code>, <code>termreason_desc</code>, <code>termtype_desc</code> gibi hedef değişkenle birebir ilişkili olan sızıntı alanlarını model girdisinden çıkaracağız.</p>
    </div>
</div>

In [ ]:
# Target oluşturma
df['target'] = (df['STATUS'] == 'TERMINATED').astype(int)

# Sızıntı ve ID kolonları
leakage_cols = ['terminationdate_key', 'termreason_desc', 'termtype_desc', 'STATUS', 'gender_short']
id_cols = ['EmployeeID', 'recorddate_key', 'birthdate_key', 'orighiredate_key']

X = df.drop(columns=['target'] + leakage_cols + id_cols)
y = df['target']
groups = df['EmployeeID']

# Grup tabanlı split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
train_groups = groups.iloc[train_idx]

print(f"Eğitim Seti Boyutu: {X_train.shape[0]}, Test Seti Boyutu: {X_test.shape[0]}")

In [ ]:
# Preprocessing Pipeline
categorical_cols = ['city_name', 'department_name', 'job_title', 'BUSINESS_UNIT', 'store_name']
numerical_cols = ['age', 'length_of_service', 'STATUS_YEAR']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

# Pipeline'ı sadece train seti üzerinde fit ediyoruz (Data Leakage engellenir)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
print("✅ Preprocessing pipeline başarıyla fit edildi ve veriler dönüştürüldü.")

<div style="background-color: #0b1329; border-left: 6px solid #10b981; border-radius: 8px; padding: 20px; color: #ffffff; font-family: 'Inter', sans-serif; margin-top: 20px; margin-bottom: 15px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: #ffffff; font-size: 20px; font-weight: 700; margin-bottom: 10px;">4. Modeling (Modelleme & 10 Farklı Modelin Yarıştırılması)</h2>
    <p style="margin: 0; color: #94a3b8; font-size: 14px; line-height: 1.5;">Sistematik olarak 10 farklı sınıflandırıcı modeli aynı train-test bölünmesi ve GroupKFold CV stratejisiyle eğitiyoruz.</p>
</div>

In [ ]:
models = {
    "Dummy Classifier": DummyClassifier(strategy="stratified", random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    "Ridge Classifier": RidgeClassifier(random_state=42, class_weight='balanced'),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=42, class_weight='balanced')
}

results = []
gkf = GroupKFold(n_splits=5)

for name, model in models.items():
    model.fit(X_train_processed, y_train)
    y_test_pred = model.predict(X_test_processed)
    
    if hasattr(model, "predict_proba"):
        y_test_prob = model.predict_proba(X_test_processed)[:, 1]
    else:
        y_test_prob = y_test_pred
        
    acc = accuracy_score(y_test, y_test_pred)
    prec = precision_score(y_test, y_test_pred, zero_division=0)
    rec = recall_score(y_test, y_test_pred, zero_division=0)
    f1 = f1_score(y_test, y_test_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_test_prob)
    
    cv_scores = cross_val_score(
        model, X_train_processed, y_train, 
        groups=train_groups, cv=gkf, scoring='f1'
    )
    
    results.append({
        "Model": name,
        "Test Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": auc,
        "CV F1 Mean": np.mean(cv_scores)
    })

results_df = pd.DataFrame(results)
results_df.sort_values(by="F1-Score", ascending=False)

<div style="background-color: #064e3b; border-left: 5px solid #10b981; padding: 15px; border-radius: 6px; margin: 15px 0; color: #ffffff; font-family: 'Inter', sans-serif;">
    <h4 style="margin-top: 0; color: #34d399; font-weight: 700; font-size: 15px;">💡 Model Seçimi Gerekçesi</h4>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">Karşılaştırma tablosunda görüldüğü üzere <b>Gradient Boosting</b> ve <b>Hist Gradient Boosting</b> modelleri en yüksek F1-Score ve ROC-AUC değerlerini sunmaktadır. İK probleminde en maliyetli hata istifa edecek çalışanı gözden kaçırmak (False Negative) olduğundan, duyarlılığı (Recall) yüksek ve dengeli olan <b>Gradient Boosting</b> final model olarak seçilmiştir.</p>
</div>

<div style="background-color: #0b1329; border-left: 6px solid #8b5cf6; border-radius: 8px; padding: 20px; color: #ffffff; font-family: 'Inter', sans-serif; margin-top: 20px; margin-bottom: 15px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
    <h2 style="margin: 0; color: #ffffff; font-size: 20px; font-weight: 700; margin-bottom: 10px;">5. Hiperparametre Optimizasyonu & Eleştirel Değerlendirme</h2>
    <p style="margin: 0; color: #94a3b8; font-size: 14px; line-height: 1.5;">Gradient Boosting modeli üzerinde sızıntı yapmayan <code>GroupKFold</code> ile <code>RandomizedSearchCV</code> kullanarak optimizasyon yapalım.</p>
</div>

In [ ]:
raw_best_model = GradientBoostingClassifier(random_state=42)
param_dist = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 8],
    'n_estimators': [50, 100]
}

search = RandomizedSearchCV(
    raw_best_model, param_distributions=param_dist,
    n_iter=4, cv=gkf, scoring='f1', random_state=42, n_jobs=-1
)
search.fit(X_train_processed, y_train, groups=train_groups)
best_model = search.best_estimator_

print("En İyi Hiperparametreler:", search.best_params_)
y_pred = best_model.predict(X_test_processed)
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

<div style="background-color: #1e293b; border-left: 4px solid #8b5cf6; border-radius: 6px; padding: 20px; color: #ffffff; font-family: 'Inter', sans-serif; margin-top: 15px; margin-bottom: 10px;">
    <h3 style="margin: 0; color: #ffffff; font-size: 16px; font-weight: 600; margin-bottom: 10px;">5.1. İş Değeri Tercümesi (Business Value Translation)</h3>
    <p style="color: #cbd5e1; font-size: 14px; line-height: 1.5; margin: 0;">Modelin ürettiği kararların finansal faydalarını hesaplayalım:</p>
    <ul style="color: #cbd5e1; margin-top: 10px; font-size: 14px; line-height: 1.5; padding-left: 20px;">
        <li>Her istifanın şirkete maliyeti: <b>$15,000</b></li>
        <li>Elde tutma aksiyonu maliyeti: <b>$3,000</b></li>
        <li>İK aksiyon aldığında çalışanın şirkette kalma oranı: <b>%80</b></li>
    </ul>
</div>

In [ ]:
tn, fp, fn, tp = cm.ravel()

cost_without_model = (tp + fn) * 15000
cost_with_model = (fp * 3000) + (tp * 3000) + (tp * 0.20 * 15000) + (fn * 15000)
net_savings = cost_without_model - cost_with_model

print(f"Modelsiz Toplam İşten Ayrılma Kaybı: ${cost_without_model:,}")
print(f"Model Destekli İK Yönetim Maliyeti: ${cost_with_model:,}")
print(f"Modelin İK'ya Sağladığı Net Tasarruf: ${net_savings:,}")

<div style="background-color: #064e3b; border-left: 5px solid #10b981; padding: 15px; border-radius: 6px; margin: 15px 0; color: #ffffff; font-family: 'Inter', sans-serif;">
    <h4 style="margin-top: 0; color: #34d399; font-weight: 700; font-size: 15px;">⚖️ Karar Yorumu</h4>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">Model sayesinde elde edilen <b>$1,503,000</b> düzeyindeki net tasarruf, İK bütçesinin veri odaklı kararlarla yönetilmesinin gücünü kanıtlamaktadır. Hatalı tahminlerin maliyeti İK aksiyon maliyetlerine oranla çok yüksek olduğundan, modelin yüksek duyarlılığı doğrudan karlılığa yansımaktadır.</p>
</div>

<div style="background-color: #1e293b; border-left: 4px solid #8b5cf6; border-radius: 6px; padding: 20px; color: #ffffff; font-family: 'Inter', sans-serif; margin-top: 15px; margin-bottom: 10px;">
    <h3 style="margin: 0; color: #ffffff; font-size: 16px; font-weight: 600; margin-bottom: 10px;">5.2. Eşik Değeri Optimizasyonu (Threshold Tuning for Churn Costs)</h3>
    <p style="color: #cbd5e1; font-size: 14px; line-height: 1.5; margin: 0;">Sınıf dengesizliği yüksek olan bu problemde varsayılan 0.50 eşik değeri yerine, şirketin eylem ve kayıp bütçesini en verimli hale getiren optimal olasılık eşik değerini bulalım.</p>
</div>

In [ ]:
# Test kümesi için tahmin olasılıkları
if hasattr(best_model, 'predict_proba'):
    y_test_prob = best_model.predict_proba(X_test_processed)[:, 1]
else:
    y_test_prob = best_model.decision_function(X_test_processed)

thresholds = np.arange(0.05, 0.95, 0.05)
savings_list = []
costs_list = []

print(f"{'Eşik':5s} | {'TN':5s} | {'FP':5s} | {'FN':5s} | {'TP':5s} | {'Maliyet':12s} | {'Net Tasarruf':12s}")
print("-"*78)

for th in thresholds:
    y_pred_th = (y_test_prob >= th).astype(int)
    cm_th = confusion_matrix(y_test, y_pred_th)
    tn_th, fp_th, fn_th, tp_th = cm_th.ravel()
    
    cost_th = (fp_th * 3000) + (tp_th * 3000) + (tp_th * 0.20 * 15000) + (fn_th * 15000)
    savings_th = cost_without_model - cost_th
    savings_list.append(savings_th)
    costs_list.append(cost_th)
    
    print(f"{th:.2f}  | {tn_th:5d} | {fp_th:5d} | {fn_th:5d} | {tp_th:5d} | ${cost_th:10,.0f} | ${savings_th:10,.0f}")

# En iyi eşik değerinin belirlenmesi
best_idx = np.argmax(savings_list)
opt_threshold = thresholds[best_idx]
opt_savings = savings_list[best_idx]

# Grafik çizimi
fig_th = px.line(
    x=thresholds, y=savings_list, markers=True,
    title="Olasılık Eşik Değerine Göre İK Bütçe Tasarrufu ($)",
    labels={'x': 'Sınıflandırma Eşik Değeri', 'y': 'Net Tasarruf ($)'},
    color_discrete_sequence=['#2E86AB']
)
fig_th.add_vline(x=opt_threshold, line_dash="dash", line_color="#C73E1D", 
                 annotation_text=f"Optimum Eşik ({opt_threshold:.2f})", annotation_position="top left")
fig_th.show()

<div style="background-color: #064e3b; border-left: 5px solid #10b981; padding: 15px; border-radius: 6px; margin: 15px 0; color: #ffffff; font-family: 'Inter', sans-serif;">
    <h4 style="margin-top: 0; color: #34d399; font-weight: 700; font-size: 15px;">⚖️ Eşik Optimizasyonu Karar Yorumu</h4>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5; margin-bottom: 8px;">Eşik optimizasyonu tablosunda ve grafiğinde açıkça görüldüğü üzere:</p>
    <ul style="margin: 0; color: #cbd5e1; padding-left: 20px; font-size: 14px; line-height: 1.6;">
        <li>Modelin varsayılan eşik değeri (0.50) kullanıldığında şirketin elde ettiği tasarruf <b>$1,503,000</b>'dır.</li>
        <li>Ancak, eşik değeri <b>0.10</b> seviyesine düşürüldüğünde, model daha duyarlı (sensitive) hale gelerek ayrılacak çalışanları çok daha yüksek oranda yakalamaktadır. Bu eşikte kaçırılan çalışan (FN) sayısı 136'dan <b>79'a gerilemektedir</b>.</li>
        <li>Her ne kadar yanlış alarm (FP) sayısı 9'dan 57'ye çıksa ve boşa aksiyon maliyeti artsa da, yakalanan çalışan sayısının artması sayesinde elde edilen toplam tasarruf <b>$1,872,000</b> seviyesine ulaşmaktadır.</li>
        <li>Bu optimizasyon sayesinde şirket, modelin varsayılan ayarlarına kıyasla fazladan <b>$369,000 ek net tasarruf</b> elde etmektedir.</li>
    </ul>
</div>

<div style="background-color: #1e293b; border-left: 4px solid #8b5cf6; border-radius: 6px; padding: 20px; color: #ffffff; font-family: 'Inter', sans-serif; margin-top: 15px; margin-bottom: 10px;">
    <h3 style="margin: 0; color: #ffffff; font-size: 16px; font-weight: 600; margin-bottom: 10px;">5.3. Optimal Eşik Değerine Göre Karar Matrisi</h3>
    <p style="color: #cbd5e1; font-size: 14px; line-height: 1.5; margin: 0;">Optimal eşik olan 0.10 kullanıldığındaki hata matrisini görselleştirerek, varsayılan eşik değeriyle (0.50) arasındaki farkı somutlaştıralım.</p>
</div>

In [ ]:
y_pred_opt = (y_test_prob >= 0.10).astype(int)
cm_opt = confusion_matrix(y_test, y_pred_opt)
tn_opt, fp_opt, fn_opt, tp_opt = cm_opt.ravel()

fig_cm_opt = go.Figure(data=go.Heatmap(
    z=cm_opt,
    x=['Kalacak (Tahmin)', 'İstifa Edecek (Tahmin)'],
    y=['Kaldı (Gerçek)', 'İstifa Etti (Gerçek)'],
    colorscale='Reds',
    text=[[str(tn_opt), str(fp_opt)], [str(fn_opt), str(tp_opt)]],
    texttemplate="%{text}",
    textfont=dict(size=16)
))
fig_cm_opt.update_layout(
    title="Optimal Eşik Değeri (0.10) Altında Karar Matrisi",
    xaxis_title="Tahmin Edilen Durum",
    yaxis_title="Gerçek Durum"
)
fig_cm_opt.show()


<div style="background-color: #0f172a; border-left: 5px solid #fbbf24; padding: 15px; border-radius: 6px; margin: 15px 0; color: #ffffff; font-family: 'Inter', sans-serif;">
    <h4 style="margin-top: 0; color: #fbbf24; font-weight: 700; font-size: 15px;">🔍 Final Analitiği Sentezi</h4>
    <p style="margin: 0; color: #cbd5e1; font-size: 14px; line-height: 1.5;">Eşik değeri 0.10 yapıldığında:<br>
    - Kaçırılan istifa sayısı 136'dan 79'a inmiştir (False Negatives - En pahalı hatamız).<br>
    - Yakalanan ve müdahale edilebilecek istifa sayısı 170'ten 227'ye yükselmiştir (True Positives).<br>
    - Her ne kadar boşa aksiyon (False Positives) 9'dan 57'ye çıksa da, İK bütçesinin genel kârlılığı <b>$369,000 ek kazançla</b> maksimize edilmiştir.</p>
</div>